<a href="https://colab.research.google.com/github/AlperYildirim1/TS-MechInterp/blob/main/SAE_Time_Series_upgraded.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q x-transformers

## Data

In [ ]:
import os
import math
import json
import random
import logging
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from x_transformers import Encoder
from tqdm.auto import tqdm
from torch.utils.tensorboard import SummaryWriter

# 0. SETUP SEED
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Device: {device}")

TARGET_RUN_NAME = "RoPE_Only_StdFFN"
PRED_LEN = 96

SAVE_DIR = '/content/drive/MyDrive/MechInterp_PatchTST'
MODEL_LOAD_DIR = '/content/drive/MyDrive/Electricity_PE_RoPE'
BEST_MODEL_PATH = os.path.join(MODEL_LOAD_DIR, f"{TARGET_RUN_NAME}_h{PRED_LEN}_seed{SEED}_best.pt")


timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

def get_logger(run_name):
    logger = logging.getLogger(run_name)
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False

    fh = logging.FileHandler(os.path.join(LOG_DIR, f"{run_name}_{timestamp}.log"))
    fh.setFormatter(logging.Formatter("%(asctime)s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S"))
    logger.addHandler(fh)

    ch = logging.StreamHandler()
    ch.setFormatter(logging.Formatter("%(message)s"))
    logger.addHandler(ch)

    return logger


print("\n⏳ Loading Electricity dataset...")
url = "https://huggingface.co/datasets/thuml/Time-Series-Library/resolve/main/electricity/electricity.csv"
df = pd.read_csv(url)

date_col = df.columns[0]
cols_data = df.drop(columns=[date_col]).values.astype(np.float32)
n_total = len(cols_data)
n_channels = cols_data.shape[1]

train_end = int(n_total * 0.7)
val_end = int(n_total * 0.8)
test_end = n_total

scaler = StandardScaler()
scaler.fit(cols_data[:train_end])
all_data = scaler.transform(cols_data)

seq_len = 336
batch_size = 64

train_slice = all_data[0:train_end]
val_slice   = all_data[train_end - seq_len : val_end]
test_slice  = all_data[val_end - seq_len : test_end]

pred_len = 96

## Classes

In [ ]:

class TSDataset(Dataset):
    def __init__(self, data, seq_len=336, pred_len=96):
        self.data = data
        self.seq_len = seq_len
        self.pred_len = pred_len
    def __len__(self):
        return len(self.data) - self.seq_len - self.pred_len + 1
    def __getitem__(self, idx):
        s = idx + self.seq_len
        return torch.FloatTensor(self.data[idx:s]), torch.FloatTensor(self.data[s:s + self.pred_len])

# 2. MODEL COMPONENTS
class RevIN(nn.Module):
    def __init__(self, num_features, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.affine_weight = nn.Parameter(torch.ones(num_features))
        self.affine_bias = nn.Parameter(torch.zeros(num_features))
    def forward(self, x, mode):
        if mode == 'norm':
            self.mean = x.mean(dim=1, keepdim=True).detach()
            self.stdev = (x.var(dim=1, keepdim=True, unbiased=False) + self.eps).sqrt().detach()
            x = (x - self.mean) / self.stdev
            return x * self.affine_weight + self.affine_bias
        else:
            x = (x - self.affine_bias) / (self.affine_weight + self.eps)
            return x * self.stdev + self.mean

class AblationPatchTST(nn.Module):
    def __init__(self, seq_len=336, pred_len=96, patch_len=16, stride=8, d_model=96,
                 transformer_depth=1, dropout=0.2, n_channels=7,
                 use_rope=True, use_w_pos=False, use_glu=True):
        super().__init__()
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.patch_len = patch_len
        self.stride = stride
        self.d_model = d_model

        # Calculate exactly how many patches we will have
        self.num_patches = (seq_len - patch_len) // stride + 2
        self.use_w_pos = use_w_pos

        # 1. Normalization
        self.revin = RevIN(n_channels)

        # 2. Patching & Projection
        self.patch_embed = nn.Linear(patch_len, d_model)
        self.pos_dropout = nn.Dropout(dropout)

        # 3. Absolute Positional Encoding (Only used if config says True)
        if self.use_w_pos:
            self.W_pos = nn.Parameter(torch.randn(1, self.num_patches, d_model) * 0.02)

        # 4. RoPE Transformer Backbone
        transformer_heads = max(1, d_model // 8) # e.g., 96 // 8 = 12 heads

        self.transformer = Encoder(
            dim=d_model,
            depth=transformer_depth,
            heads=transformer_heads,
            attn_dim_head=16,
            ff_mult=2,                # expansion (192 inner dim)
            attn_flash=True,
            rotary_pos_emb=use_rope,
            use_rmsnorm=True,
            ff_glu=use_glu,
            attn_dropout=dropout,
            ff_dropout=dropout
        )

        # 5. Prediction Head
        self.flatten_dim = self.num_patches * d_model
        self.head_drop = nn.Dropout(dropout)
        self.head = nn.Linear(self.flatten_dim, pred_len)

    def forward(self, x):
        x = self.revin(x, 'norm')
        B, L, C = x.shape
        x = x.permute(0, 2, 1).reshape(B * C, L, 1)

        x = x.transpose(1, 2)
        x = F.pad(x, (0, self.stride), 'replicate')
        patches = x.unfold(-1, self.patch_len, self.stride).squeeze(1)

        raw_embed = self.patch_embed(patches)

        if self.use_w_pos:
            features = self.pos_dropout(raw_embed + self.W_pos[:, :raw_embed.shape[1], :])
        else:
            features = self.pos_dropout(raw_embed)

        features = self.transformer(features)

        # Flatten and predict
        flattened = self.head_drop(features.reshape(features.shape[0], -1))
        out = self.head(flattened).unsqueeze(-1)

        # Reshape back to multivariate format and De-Normalize
        out = out.reshape(B, C, self.pred_len).permute(0, 2, 1)
        return self.revin(out, 'denorm')

## Setup and Load

In [ ]:
# 1. SETUP & LOAD THE BEST MODEL

print(f"Loading best model from: {BEST_MODEL_PATH}")

model = AblationPatchTST(
    seq_len=seq_len,
    pred_len=pred_len,
    d_model=128,
    transformer_depth=1,
    dropout=0.2,
    n_channels=n_channels,
    use_rope=True,
    use_w_pos=False,
    use_glu=False  # Standard FFN
).to(device)

model.load_state_dict(torch.load(BEST_MODEL_PATH))
model.eval()
for param in model.parameters():
    param.requires_grad = False

# Re-create the DataLoader
g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    TSDataset(train_slice, seq_len, pred_len),
    batch_size=batch_size, shuffle=True, drop_last=True,
    generator=g, worker_init_fn=lambda w: np.random.seed(SEED + w),
    num_workers=2, pin_memory=True
)

## Activations

In [ ]:
"""
This code dynamicly searches for GELU. A bit overengineering but it works fine.
If x-transformers gets an update, it should still work as long as we use GELU.

Our model is trained bf16, so this also converts it to float32 for SAE
"""

# 2. HARVEST THE ACTIVATIONS
activation_buffer =[]

def get_activation_hook():
    def hook(module, input, output):
        # We immediately cast to float16 and move to CPU to save RAM
        activation_buffer.append(output.detach().cpu().half())
    return hook

# Search the model for the GELU layer
target_module = None
for name, module in model.named_modules():
    if isinstance(module, nn.GELU):
        target_module = module
        print(f" Successfully found GELU at: {name}")
        break

if target_module is None:
    raise ValueError("Could not find the GELU activation! Please check the model structure.")

# Attach the hook
hook_handle = target_module.register_forward_hook(get_activation_hook())

print("\n🚜 Harvesting FFN Activations from Training Data...")

# The RAM Cap (Prevents the 250GB explosion)
MAX_PATCHES = 2_000_000
total_patches_harvested = 0

with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
    for x, _ in tqdm(train_loader, desc="Forward Pass"):
        x = x.to(device)
        _ = model(x)

        last_shape = activation_buffer[-1].shape
        # Flatten batch and sequence to count total patches
        total_patches_harvested += (last_shape[0] * last_shape[1])

        if total_patches_harvested >= MAX_PATCHES:
            print(f"\n Reached patch limit ({total_patches_harvested}). Stopping harvest safely.")
            break

# Remove hook so it doesn't leak memory later
hook_handle.remove()

# Stack everything into one massive 2D tensor and convert to float32 for SAE
all_activations = torch.cat(activation_buffer, dim=0)
all_activations = all_activations.reshape(-1, all_activations.shape[-1]).float()

print(f" Harvest Complete! Tensor Shape: {all_activations.shape}")

## Build the Sparse Autoencoder

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import os
from tqdm.auto import tqdm

class SparseAutoencoder(nn.Module):
    def __init__(self, d_in, d_hidden):
        super().__init__()
        # Encoder: maps FFN activation up to the dictionary size
        self.encoder = nn.Linear(d_in, d_hidden)
        # Decoder: maps back down to reconstruct the original FFN activation
        self.decoder = nn.Linear(d_hidden, d_in)

        # Initialize decoder weights to be normalized
        with torch.no_grad():
            self.decoder.weight.data = F.normalize(self.decoder.weight.data, dim=0)

    def forward(self, x):
        # We use ReLU to enforce absolute dead-zero sparsity
        f = F.relu(self.encoder(x))
        x_reconstructed = self.decoder(f)
        return x_reconstructed, f

# SAE Hyperparameters
SAE_EXPANSION_FACTOR = 4
d_in = 256  # Inner dim
d_hidden = d_in * SAE_EXPANSION_FACTOR # 1024 latents

sae = SparseAutoencoder(d_in=d_in, d_hidden=d_hidden).to(device)
sae_optimizer = torch.optim.Adam(sae.parameters(), lr=1e-3)

# The most important knob: L1 Penalty
L1_COEFF = 0.01

sae_loader = DataLoader(
    TensorDataset(all_activations),
    batch_size=8192,  # We can go huge here because it's just a 1-layer MLP
    shuffle=True,
    pin_memory=True
)


## Training

In [ ]:

# 4. TRAIN THE SAE
epochs = 5  # Start with 5 epochs to see how L0 behaves
print(f"\n Training SAE | Latents: {d_hidden} | L1 Coeff: {L1_COEFF}")

sae.train()
for epoch in range(epochs):
    total_loss, total_mse, total_l1 = 0, 0, 0
    total_l0 = 0 # L0 = How many latents are firing > 0 per patch

    pbar = tqdm(sae_loader, desc=f"SAE Epoch {epoch+1}")
    for (batch_x,) in pbar:
        batch_x = batch_x.to(device) # Move to GPU

        sae_optimizer.zero_grad()

        x_recon, f = sae(batch_x)

        # 1. Reconstruction Loss (MSE)
        mse_loss = F.mse_loss(x_recon, batch_x)

        # 2. Sparsity Loss (L1)
        l1_loss = f.abs().sum(dim=-1).mean()

        # Total Loss
        loss = mse_loss + (L1_COEFF * l1_loss)

        loss.backward()
        sae_optimizer.step()

        # Re-normalize decoder weights
        with torch.no_grad():
            sae.decoder.weight.data = F.normalize(sae.decoder.weight.data, dim=0)

        total_loss += loss.item()
        total_mse += mse_loss.item()
        total_l1 += l1_loss.item()
        total_l0 += (f > 0).float().sum(dim=-1).mean().item()

        pbar.set_postfix({
            'MSE': f"{mse_loss.item():.4f}",
            'L0': f"{(f > 0).float().sum(dim=-1).mean().item():.1f}"
        })

    avg_l0 = total_l0 / len(sae_loader)
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(sae_loader):.4f} | MSE: {total_mse/len(sae_loader):.4f} | Avg L0 (Active Latents): {avg_l0:.1f}")

# Save the trained SAE
SAE_SAVE_PATH = os.path.join(SAVE_DIR, f"SAE_Electricity_{d_hidden}latents_L1_{L1_COEFF}.pt")
torch.save(sae.state_dict(), SAE_SAVE_PATH)
print(f"SAE saved to {SAE_SAVE_PATH}")

## Downstream Loss

In [ ]:
# 5. SANITY CHECK: Downstream Loss Fidelity
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

print("🔬 Running Downstream Fidelity Sanity Check...")

# 1. Rebuild the Test Loader (No shuffling needed for testing)
test_loader = DataLoader(
    TSDataset(test_slice, seq_len, pred_len),
    batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True
)

model.eval()
sae.eval()

# 2. Find the exact GELU layer again (just to be safe)
target_module = None
for name, module in model.named_modules():
    if isinstance(module, nn.GELU):
        target_module = module
        break

if target_module is None:
    raise ValueError("Could not find the GELU activation!")

# TEST A: BASELINE FORECASTING
base_mse_sum, base_mae_sum, total_samples = 0.0, 0.0, 0

with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
    for x, y in tqdm(test_loader, desc="Baseline Test"):
        x, y = x.to(device), y.to(device)
        preds = model(x)

        b_size = x.size(0)
        base_mse_sum += F.mse_loss(preds, y).item() * b_size
        base_mae_sum += F.l1_loss(preds, y).item() * b_size
        total_samples += b_size

base_mse = base_mse_sum / total_samples
base_mae = base_mae_sum / total_samples

# TEST B: SAE-INTERVENED FORECASTING
# This hook intercepts the GELU output, runs it through the SAE, and returns the reconstruction
def get_sae_intervention_hook():
    def hook(module, input, output):
        orig_shape = output.shape
        orig_dtype = output.dtype

        # Flatten and convert to float32 for the SAE
        flat_x = output.view(-1, orig_shape[-1]).float()

        # Reconstruct using our 7 sparse latents!
        x_recon, _ = sae(flat_x)

        # Reshape, cast back to original dtype, and inject back into the network
        return x_recon.view(orig_shape).to(orig_dtype)
    return hook

# Attach the brain transplant hook
intervention_hook = target_module.register_forward_hook(get_sae_intervention_hook())

sae_mse_sum, sae_mae_sum = 0.0, 0.0

with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
    for x, y in tqdm(test_loader, desc="SAE Intervened Test"):
        x, y = x.to(device), y.to(device)
        preds = model(x)  # The hook automatically intercepts and modifies the thoughts!

        b_size = x.size(0)
        sae_mse_sum += F.mse_loss(preds, y).item() * b_size
        sae_mae_sum += F.l1_loss(preds, y).item() * b_size

sae_mse = sae_mse_sum / total_samples
sae_mae = sae_mae_sum / total_samples

# Remove the hook
intervention_hook.remove()

# RESULTS
print("\n" + "="*50)
print(" DOWNSTREAM FIDELITY RESULTS")
print("="*50)
print(f"Original Baseline Model | Test MSE: {base_mse:.4f} | Test MAE: {base_mae:.4f}")
print(f"SAE-Intervened Model    | Test MSE: {sae_mse:.4f} | Test MAE: {sae_mae:.4f}")
print("="*50)

mse_degradation = (sae_mse - base_mse) / base_mse * 100
print(f"Performance Degradation: {mse_degradation:.2f}%")

## Visualise

In [ ]:
# 6. THE COMPLETE DICTIONARY: LOUDEST vs. MOST FREQUENT
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import os

print("🔎 Hunting 5,000+ building sensors for BOTH specialized and everyday motifs...")

model.eval()
sae.eval()

# 1. Grab 16 batches, keeping all 321 channels for RevIN safety
x_sample, y_sample = next(iter(test_loader))
x_subset = x_sample[:16, :, :].to(device)

latents_buffer =[]

def get_sae_features_hook(module, input, output):
    flat_x = output.view(-1, output.shape[-1]).float()
    _, f = sae(flat_x)
    latents_buffer.append(f.detach().float().cpu())

target_module = None
for name, module in model.named_modules():
    if isinstance(module, nn.GELU):
        target_module = module
        break

hook_handle = target_module.register_forward_hook(get_sae_features_hook)

with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
    _ = model(x_subset)

hook_handle.remove()

latents = latents_buffer[0]


# CALCULATE THE TWO METRICS
# Metric A: Loudest Magnitude (The Anomaly / Event Detectors)
max_acts_per_latent = latents.max(dim=0)[0]
top_mag_latents = torch.argsort(max_acts_per_latent, descending=True)[:3]

# Metric B: Most Frequent (The Everyday Grammar / Cycles)
# We use a tiny threshold (0.01) to ignore microscopic floating-point noise
freq_per_latent = (latents > 0.01).float().sum(dim=0)
top_freq_latents = torch.argsort(freq_per_latent, descending=True)[:3]

print(f"\n🔥 Top 3 LOUDEST Latents: {top_mag_latents.tolist()}")
print(f"🌊 Top 3 MOST FREQUENT Latents: {top_freq_latents.tolist()}\n")

# REUSABLE PLOTTING FUNCTION
def plot_latent(latent_idx, category_name):
    # Find the sequence where this latent is most visible so we can see its shape clearly
    best_patch_idx = torch.argmax(latents[:, latent_idx]).item()

    seq_idx = best_patch_idx // 42
    b_idx = seq_idx // 321
    c_idx = seq_idx % 321

    raw_ts = x_subset[b_idx, :, c_idx].cpu().numpy()

    start_row = seq_idx * 42
    end_row = start_row + 42
    seq_activations = latents[start_row:end_row, latent_idx].numpy()

    seq_len = 336
    patch_len = 16
    stride = 8

    expanded_activations = np.zeros(seq_len)
    for p, act in enumerate(seq_activations):
        s_idx = p * stride
        e_idx = min(s_idx + patch_len, seq_len)
        expanded_activations[s_idx:e_idx] += act

    fig, ax1 = plt.subplots(figsize=(14, 4))

    line1 = ax1.plot(raw_ts, color='black', linewidth=1.5, label=f"Sensor Wave (Batch {b_idx}, Ch {c_idx})", zorder=2)
    ax1.set_xlim(0, seq_len)
    ax1.set_ylabel("Normalized Usage")
    ax1.grid(alpha=0.2)

    ax2 = ax1.twinx()
    line2 = ax2.plot(expanded_activations, color='red', linewidth=2, alpha=0.8, label=f"Latent #{latent_idx} Activation")
    ax2.fill_between(range(seq_len), 0, expanded_activations, color='red', alpha=0.2)

    ax2.set_ylabel("SAE Activation Strength", color='red')
    ax2.tick_params(axis='y', labelcolor='red')

    ymax = max(expanded_activations)
    ax2.set_ylim(0, ymax * 1.5 if ymax > 0 else 1.0)

    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc="upper right")

    plt.title(f"[{category_name}] What triggers SAE Latent #{latent_idx}?")
    plt.tight_layout()

    # Save the high-res figure directly to your Drive!
    filename = f"SAE_Latent_{latent_idx}_{category_name.replace(' ', '_')}.png"
    plot_path = os.path.join(SAVE_DIR, filename)
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f"💾 Saved: {filename}")

    plt.show()

# GENERATE THE PLOTS
print("=== PLOTTING LOUDEST (MAGNITUDE) MOTIFS ===")
for l_idx in top_mag_latents:
    plot_latent(l_idx.item(), "Highest Magnitude")

print("=== PLOTTING MOST FREQUENT (EVERYDAY) MOTIFS ===")
for l_idx in top_freq_latents:
    plot_latent(l_idx.item(), "Most Frequent")

## Dead Latent Analysis

In [ ]:
import torch
import torch.nn as nn
from tqdm.auto import tqdm

print("💀 Investigating Dead Latents across the entire Test Set...")

model.eval()
sae.eval()

# Create a boolean mask to track if a latent ever fires
total_dictionary_size = sae.encoder.out_features
latents_fired = torch.zeros(total_dictionary_size, device=device, dtype=torch.bool)

def dead_latent_hook(module, input, output):
    flat_x = output.view(-1, output.shape[-1]).float()
    _, f = sae(flat_x)

    # If a latent activates > 0 anywhere in this batch, mark it as True (Alive)
    global latents_fired
    latents_fired |= (f > 1e-5).any(dim=0)

# Find GELU and attach
target_module = None
for name, module in model.named_modules():
    if isinstance(module, nn.GELU):
        target_module = module
        break

if target_module is None:
    raise ValueError("Could not find the GELU activation!")

hook_handle = target_module.register_forward_hook(dead_latent_hook)

# Run the entire test set
with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
    for x, _ in tqdm(test_loader, desc="Scanning for Active Motifs"):
        x = x.to(device)
        _ = model(x)

hook_handle.remove()

# Calculate results
alive_count = latents_fired.sum().item()
dead_count = total_dictionary_size - alive_count

print("\n" + "="*50)
print(" 🧟 DICTIONARY HEALTH REPORT")
print("="*50)
print(f"Total Dictionary Size: {total_dictionary_size}")
print(f"Alive Latents:         {alive_count}")
print(f"Dead Latents:          {dead_count} ({(dead_count/total_dictionary_size)*100:.2f}%)")
print("="*50)

if dead_count > (total_dictionary_size * 0.5):
    print("\n INCREDIBLE RESULT FOR THE PAPER:")
    print("The majority of the dictionary is dead! This conclusively proves that")
    print("time series data has massively lower intrinsic dimensionality than text.")
else:
    print("\nInteresting! The dataset actually utilizes a wide variety of the dictionary.")

## Superposition Test

In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

print("🔬 Testing for True Superposition: Reconstruction vs. Downstream Forecasting...")

# 1. Ensure test_loader is ready
test_loader = DataLoader(
    TSDataset(test_slice, seq_len, pred_len),
    batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True
)

# 2. Find the GELU layer
target_module = None
for name, module in model.named_modules():
    if isinstance(module, nn.GELU):
        target_module = module
        break

if target_module is None:
    raise ValueError("Could not find the GELU activation!")

# 3. Calculate True Baseline Forecasting MSE
print("📊 Calculating Baseline Forecasting MSE...")
base_mse_sum, total_samples = 0.0, 0
with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        preds = model(x)
        b_size = x.size(0)
        base_mse_sum += F.mse_loss(preds, y).item() * b_size
        total_samples += b_size

base_mse = base_mse_sum / total_samples
print(f"✅ Baseline Test MSE: {base_mse:.4f}\n")

# 4. Train SAEs and Test Fidelity
d_in = 256
expansion_factors = [0.5, 1.0, 4.0]  # Latent counts: 128, 256, 1024
L1_COEFF = 0.01
epochs = 3

results = {}

for factor in expansion_factors:
    d_hidden = int(d_in * factor)
    print(f"🚀 Training & Testing SAE with {d_hidden} Latents (Expansion: {factor}x)")

    # --- A. TRAIN SAE ---
    sae_test = SparseAutoencoder(d_in, d_hidden).to(device)
    sae_optimizer = torch.optim.Adam(sae_test.parameters(), lr=1e-3)

    sae_loader = DataLoader(
        TensorDataset(all_activations),
        batch_size=8192, shuffle=True, pin_memory=True
    )

    sae_test.train()
    for epoch in range(epochs):
        total_mse, total_l0 = 0, 0
        for (batch_x,) in sae_loader:
            batch_x = batch_x.to(device)
            sae_optimizer.zero_grad()

            x_recon, f = sae_test(batch_x)
            mse_loss = F.mse_loss(x_recon, batch_x)
            l1_loss = f.abs().sum(dim=-1).mean()
            loss = mse_loss + (L1_COEFF * l1_loss)

            loss.backward()
            sae_optimizer.step()

            with torch.no_grad():
                sae_test.decoder.weight.data = F.normalize(sae_test.decoder.weight.data, dim=0)

            total_mse += mse_loss.item()
            total_l0 += (f > 0).float().sum(dim=-1).mean().item()

        avg_recon_mse = total_mse / len(sae_loader)
        avg_l0 = total_l0 / len(sae_loader)

    # --- B. TEST DOWNSTREAM FORECASTING FIDELITY ---
    sae_test.eval()

    def get_sae_intervention_hook(current_sae):
        def hook(module, input, output):
            orig_shape = output.shape
            orig_dtype = output.dtype
            flat_x = output.view(-1, orig_shape[-1]).float()
            x_recon, _ = current_sae(flat_x)
            return x_recon.view(orig_shape).to(orig_dtype)
        return hook

    intervention_hook = target_module.register_forward_hook(get_sae_intervention_hook(sae_test))

    sae_forecast_mse_sum = 0.0
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            preds = model(x)
            sae_forecast_mse_sum += F.mse_loss(preds, y).item() * x.size(0)

    intervention_hook.remove()

    forecast_mse = sae_forecast_mse_sum / total_samples
    degradation = ((forecast_mse - base_mse) / base_mse) * 100

    print(f"   ↳ Recon MSE: {avg_recon_mse:.4f} | L0: {avg_l0:.1f} | Forecast MSE: {forecast_mse:.4f} ({degradation:+.2f}%)\n")

    results[d_hidden] = {
        "Recon_MSE": avg_recon_mse,
        "L0": avg_l0,
        "Forecast_MSE": forecast_mse,
        "Degradation": degradation
    }

# ==========================================
# SUMMARY TABLE (FOR THE PAPER)
# ==========================================
print("="*80)
print(f" 📊 THE SUPERPOSITION & FIDELITY REPORT  (Baseline Forecast MSE: {base_mse:.4f})")
print("="*80)
print(f"{'Dictionary Size':<18} | {'SAE Recon MSE':<15} | {'L0 (Sparsity)':<15} | {'Forecast MSE':<15} | {'Degradation'}")
print("-" * 80)
for size, m in results.items():
    print(f"{size:<18} | {m['Recon_MSE']:<15.4f} | {m['L0']:<15.1f} | {m['Forecast_MSE']:<15.4f} | {m['Degradation']:+.2f}%")
print("="*80)

## Nearest Neighbor

In [ ]:
import matplotlib.pyplot as plt
import torch

print("📐 Running Decoder Geometry & Nearest Neighbor Analysis...")

# 1. Extract and normalize the decoder weights
W = sae.decoder.weight.data.detach() # Shape: [256, 1024]

# 2. Filter down to ONLY the alive latents
alive_W = W[:, latents_fired] if 'latents_fired' in locals() else W
alive_W = torch.nn.functional.normalize(alive_W, dim=0)

# 3. Compute pairwise cosine similarity matrix
cos_sim_matrix = alive_W.T @ alive_W  # Shape: [Alive, Alive]

# 4. Extract Nearest Neighbors (Max Cosine per latent)
cos_sim_matrix.fill_diagonal_(-1.0)
max_cosines, _ = cos_sim_matrix.max(dim=1)

# 5. Extract all unique pairs (Upper Triangle)
n = cos_sim_matrix.shape[0]
triu_indices = torch.triu_indices(n, n, offset=1)
all_pairwise_cosines = cos_sim_matrix[triu_indices[0], triu_indices[1]]

# 6. Plot the results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

ax1.hist(all_pairwise_cosines.cpu().numpy(), bins=100, color='blue', alpha=0.7)
ax1.set_title("Global Pairwise Cosine Similarity")
ax1.set_xlabel("Cosine Similarity")
ax1.set_ylabel("Count of Pairs")
ax1.grid(alpha=0.3)

ax2.hist(max_cosines.cpu().numpy(), bins=50, color='red', alpha=0.7)
ax2.set_title("Nearest Neighbor Similarity (Max Cosine per Latent)")
ax2.set_xlabel("Max Cosine Similarity")
ax2.set_ylabel("Count of Latents")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"📊 Mean Nearest Neighbor Similarity: {max_cosines.mean().item():.4f}")
print(f"⚠️ Latents with >0.8 similarity (High Splitting): {(max_cosines > 0.8).sum().item()}")

## Intervention Sweep Loudest

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

print("🧪 Running Causal Intervention Sweep...")

# 1. Setup Parameters
# Let's use the #1 Loudest Latent
target_latent_idx = top_mag_latents[0].item() if 'top_mag_latents' in locals() else 0
scales = [0.0, 0.5, 1.0, 2.0, 5.0]

# Grab exactly ONE batch for testing
x_sample, y_true = next(iter(test_loader))
x_sample = x_sample.to(device)
B, Seq_Len, C = x_sample.shape

# ==========================================
# NEW STEP: The Radar (Find where the latent is active)
# ==========================================
print(f"📡 Scanning batch to find where Latent #{target_latent_idx} is most active...")
activations_for_batch = None

def get_peek_hook():
    def hook(module, input, output):
        global activations_for_batch
        orig_shape = output.shape  # [B*C, Num_Patches, d_model]
        flat_x = output.view(-1, orig_shape[-1]).float()

        # Forward through SAE to get latents
        _, f = sae(flat_x)

        # Reshape back to [B, C, Num_Patches, d_hidden]
        Num_Patches = orig_shape[1]
        f_reshaped = f.view(B, C, Num_Patches, -1)

        # Extract just our target latent and bring to CPU
        activations_for_batch = f_reshaped[:, :, :, target_latent_idx].detach().cpu()
    return hook

target_module = None
for name, module in model.named_modules():
    if isinstance(module, torch.nn.GELU):
        target_module = module
        break

# Run radar pass
radar_hook = target_module.register_forward_hook(get_peek_hook())
with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
    _ = model(x_sample)
radar_hook.remove()

# Sum the activations across all patches in each sequence to find the "loudest" sequence
act_sum = activations_for_batch.sum(dim=-1)  # Shape: [B, C]
flat_max_idx = torch.argmax(act_sum).item()
best_b_idx = flat_max_idx // C
best_c_idx = flat_max_idx % C
max_act_val = act_sum[best_b_idx, best_c_idx].item()

if max_act_val == 0:
    print("⚠️ WARNING: This latent did not fire AT ALL in this entire batch. The plot will be flat.")
    print("Try running the cell again to grab a new batch, or change 'target_latent_idx'.")
else:
    print(f"🎯 Found target! Highest activity is at Batch {best_b_idx}, Channel {best_c_idx} (Total Act: {max_act_val:.2f})")

original_input = x_sample[best_b_idx, :, best_c_idx].cpu().numpy()

# ==========================================
# THE INTERVENTION SWEEP
# ==========================================
def get_scaling_hook(latent_idx, scale_factor):
    def hook(module, input, output):
        orig_shape = output.shape
        orig_dtype = output.dtype
        flat_x = output.view(-1, orig_shape[-1]).float()

        # SAE forward pass
        f = torch.nn.functional.relu(sae.encoder(flat_x))

        # INTERVENTION: Scale the target latent
        f[:, latent_idx] = f[:, latent_idx] * scale_factor

        # Reconstruct and inject
        x_recon = sae.decoder(f)
        return x_recon.view(orig_shape).to(orig_dtype)
    return hook

results = {}

model.eval()
print("🔬 Running geometric sweeps...")
with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
    for scale in scales:
        hook_handle = target_module.register_forward_hook(get_scaling_hook(target_latent_idx, scale))

        preds = model(x_sample)

        # Extract the exact prediction we identified in the radar step
        results[scale] = preds[best_b_idx, :, best_c_idx].cpu().numpy()

        hook_handle.remove()

# ==========================================
# PLOTTING
# ==========================================
plt.figure(figsize=(14, 6))

# Plot the historical context
plt.plot(range(Seq_Len), original_input, color='black', label=f"Historical Input (Ch {best_c_idx})", linewidth=2)

# Plot the swept forecasts
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(scales)))
for i, scale in enumerate(scales):
    plt.plot(
        range(Seq_Len, Seq_Len + 96),
        results[scale],
        color=colors[i],
        alpha=0.8,
        linewidth=2,
        label=f"Forecast (Latent x{scale})"
    )

plt.title(f"Causal Intervention Sweep on Latent #{target_latent_idx}")
plt.xlabel("Time Steps")
plt.ylabel("Normalized Electricity Value")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Else

In [ ]:
LOG_DIR = SAVE_DIR
# ENV LOGGING
import subprocess, platform, sys

env_info = {
    "python": sys.version,
    "platform": platform.platform(),
    "cpu": platform.processor(),
    "pytorch": torch.__version__,
}

if torch.cuda.is_available():
    env_info["cuda_version"] = torch.version.cuda
    env_info["cudnn_version"] = str(torch.backends.cudnn.version())
    env_info["gpu_name"] = torch.cuda.get_device_name(0)
    env_info["gpu_memory_gb"] = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
    env_info["gpu_count"] = torch.cuda.device_count()
else:
    env_info["gpu"] = "None (CPU only)"

try:
    freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
    env_info["pip_freeze"] = freeze.strip().split("\n")
except Exception:
    env_info["pip_freeze"] = "unavailable"

env_path = os.path.join(LOG_DIR, f"environment_{timestamp}.json")
with open(env_path, 'w') as f:
    json.dump(env_info, f, indent=2)

print(json.dumps({k: v for k, v in env_info.items() if k != "pip_freeze"}, indent=2))
print(f"\nFull env saved to: {env_path}")

In [ ]:
import time
from google.colab import runtime

# Sleep for 30 seconds
time.sleep(30)

# Unassign and disconnect the runtime
runtime.unassign()